# Resample TotalSegmentator masks by gender and split


In [ ]:
import glob
import os
import pandas as pd
import SimpleITK as sitk
import numpy as np

DATA_SPLIT = "train_val"  # options: "train_val", "test"
GENDER = "male"  # options: "male", "female"

BASE_DIR = r"/path/to/workspace/classification_model_originalimage"
IMAGE_DIRS = {
    "train_val": ["train_data", "val_data"],
    "test": ["test_data"],
}
GENDER_CSV = {
    "train_val": "Gender_find.csv",
    "test": "Gender_find_test.csv",
}

image_paths = []
for subdir in IMAGE_DIRS[DATA_SPLIT]:
    image_paths.extend(glob.glob(os.path.join(BASE_DIR, subdir, "*.nii.gz")))

gender_csv_path = os.path.join(BASE_DIR, "totalsegment_work", "resampled", GENDER_CSV[DATA_SPLIT])
df = pd.read_csv(gender_csv_path)
save_path = os.path.join(BASE_DIR, "totalsegment_work", "resampled", f"{GENDER}_{DATA_SPLIT}")
print(DATA_SPLIT, GENDER, len(image_paths), save_path)


In [ ]:
def resample_image(img, target_spacing, is_label=False):
    original_spacing = img.GetSpacing()
    original_size = img.GetSize()
    new_size = [
        int(np.round(original_size[i] * (original_spacing[i] / target_spacing[i])))
        for i in range(3)
    ]

    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputDirection(img.GetDirection())
    resampler.SetOutputOrigin(img.GetOrigin())
    resampler.SetInterpolator(sitk.sitkNearestNeighbor if is_label else sitk.sitkLinear)
    return resampler.Execute(img)


def generate_path(image_path, mask_chosen):
    filename = os.path.basename(image_path)
    patient_number = filename[:4]
    return os.path.join(BASE_DIR, "totalsegment_work", patient_number, mask_chosen)


In [ ]:
target_spacing = (1.0, 1.0, 1.0)
os.makedirs(save_path, exist_ok=True)

for img_path in image_paths:
    patient_number = os.path.basename(img_path)[:4]

    # Adjust these column names if your gender CSV uses different labels.
    row = df[df.astype(str).apply(lambda col: col.str.contains(patient_number, na=False)).any(axis=1)]
    if row.empty:
        continue
    if not row.astype(str).apply(lambda col: col.str.lower().str.contains(GENDER, na=False)).any(axis=1).iloc[0]:
        continue

    image = sitk.ReadImage(img_path)
    colon_image = sitk.ReadImage(generate_path(img_path, "colon.nii.gz"))
    resampled_image = resample_image(image, target_spacing)
    resampled_colon = resample_image(colon_image, target_spacing, is_label=True)

    new_folder = os.path.join(save_path, patient_number)
    os.makedirs(new_folder, exist_ok=True)
    sitk.WriteImage(resampled_image, os.path.join(new_folder, "resampled_image.nii.gz"))
    sitk.WriteImage(resampled_colon, os.path.join(new_folder, "colon.nii.gz"))

    if GENDER == "male":
        prostate_image = sitk.ReadImage(generate_path(img_path, "prostate.nii.gz"))
        resampled_prostate = resample_image(prostate_image, target_spacing, is_label=True)
        sitk.WriteImage(resampled_prostate, os.path.join(new_folder, "prostate.nii.gz"))

print("done")
